Overall population of erf and gloopers over ticks:

In [ ]:
# --- CONFIGURATION ---
target_run_id = 1
# ---------------------

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('dark_background')
conn = sqlite3.connect("data/ecosystem.db")

# This query selects only the MOST RECENT entry (MAX(id)) for each tick_number
# to prevent "spikes" caused by resuming or duplicate saves.
query = f"""
    WITH unique_ticks AS (
        SELECT MAX(id) as latest_id, tick_number
        FROM ticks
        WHERE run_id = {target_run_id}
        GROUP BY tick_number
    )
    SELECT 
        ut.tick_number, 
        LOWER(cs.species) as species, 
        COUNT(cs.id) as count
    FROM unique_ticks ut
    JOIN creature_states cs ON ut.latest_id = cs.tick_id
    GROUP BY ut.tick_number, LOWER(cs.species)
    ORDER BY ut.tick_number
"""

df = pd.read_sql_query(query, conn)

if not df.empty:
    # Pivot the data so 'erf' and 'glooper' are columns
    pivot_df = df.pivot(index='tick_number', columns='species', values='count').fillna(0)
    
    plt.figure(figsize=(14, 7))
    
    # Define colors
    colors = {'erf': '#00BFFF', 'glooper': '#00FF00'} # Blue for Erfs, Green for Gloopers
    
    for species in pivot_df.columns:
        plt.plot(pivot_df.index, pivot_df[species], 
                 label=f'{species.capitalize()} Population', 
                 linewidth=2.5, 
                 color=colors.get(species, None))

    plt.xlabel('Tick Number', fontsize=18)
    plt.ylabel('Population', fontsize=18)
    plt.title(f'Predator-Prey Population - Run {target_run_id}', fontsize=20, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print(f"No creature data found for Run ID: {target_run_id}")

male/female erf population over ticks:

In [ ]:
# Separate plot cell for male/female population over ticks
# The ticks table only stores total population, so we derive sex counts from creature_states.
# Some runs contain duplicated tick_number values after resuming, which can artificially inflate counts.

# Apply dark theme and larger fonts for poster visibility
plt.style.use('dark_background')
plt.rcParams.update({
    'font.size': 16,
    'axes.labelsize': 18,
    'axes.titlesize': 20,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 16,
    'figure.titlesize': 22
})

query = '''
SELECT t.tick_number,
       cs.sex,
       COUNT(*) AS count
FROM creature_states cs
JOIN (
    SELECT tick_number, MAX(id) AS id
    FROM ticks
    WHERE run_id = ?
    GROUP BY tick_number
) t ON cs.tick_id = t.id
GROUP BY t.tick_number, cs.sex
ORDER BY t.tick_number, cs.sex
'''
gender_df = pd.read_sql_query(query, conn, params=(target_run_id,))

if gender_df.empty:
    print('No creature sex data found for this run. The ticks table only stores total population.')
else:
    pivot_df = gender_df.pivot(index='tick_number', columns='sex', values='count').fillna(0)
    pivot_df = pivot_df.rename(columns={
        'M': 'Males',
        'F': 'Females',
    })

    plt.figure(figsize=(max(14, len(pivot_df) / 50), 7))
    if 'Males' in pivot_df.columns:
        plt.plot(pivot_df.index, pivot_df['Males'], label='Males', color='#1f77b4', linewidth=2.5)
    if 'Females' in pivot_df.columns:
        plt.plot(pivot_df.index, pivot_df['Females'], label='Females', color='#e377c2', linewidth=2.5)

    plt.xlabel('Tick Number', fontsize=18)
    plt.ylabel('Creature Count', fontsize=18)
    plt.title(f'Male vs Female Population Over Time - Run {target_run_id}', fontsize=20, fontweight='bold')
    plt.legend(fontsize=16)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


Geographic heatmap of the erfs

In [ ]:
import seaborn as sns

# 1. Configuration
# target_run_id should be defined in your first cell
# conn should be your active sqlite3 connection

# 2. Optimized Query using your specific column names: pos_x, pos_y
query = '''
SELECT cs.pos_x, cs.pos_y 
FROM creature_states cs
JOIN ticks t ON cs.tick_id = t.id
WHERE t.run_id = ? 
  AND t.tick_number = (SELECT MAX(tick_number) FROM ticks WHERE run_id = ?)
  AND cs.alive = 1
'''

try:
    pos_df = pd.read_sql_query(query, conn, params=(target_run_id, target_run_id))

    if not pos_df.empty:
        # 3. Create the Density Plot
        plt.figure(figsize=(12, 10))
        
        # 'mako' or 'viridis' look great on dark backgrounds
        # fill=True creates the solid "heat" effect
        sns.kdeplot(
            data=pos_df, x='pos_x', y='pos_y', 
            fill=True, 
            thresh=0, 
            levels=100, 
            cmap='mako',
            cbar=True,
            cbar_kws={'label': 'Creature Density'}
        )

        # 4. Styling
        plt.title(f'Final Tick Concentration Heatmap (Run {target_run_id})', fontsize=20, pad=20)
        plt.xlabel('World X Coordinate', fontsize=14)
        plt.ylabel('World Y Coordinate', fontsize=14)
        
        # Adjust these limits based on your Godot world size
        plt.xlim(pos_df['pos_x'].min() - 10, pos_df['pos_y'].max() + 10) 
        plt.ylim(pos_df['pos_y'].min() - 10, pos_df['pos_y'].max() + 10)

        plt.tight_layout()
        plt.show()
    else:
        print(f"No living creatures found at the end of Run {target_run_id} to map.")

except Exception as e:
    print(f"Error generating heatmap: {e}")